# Real 2 m temperature — styled plot **and animation**

`examples/data/europe_t2m.npz` is a real **ERA5 2 m air temperature** stack over Europe: 45 daily frames
(°C). It is rendered with the `temperature_2m` style (ECMWF's `Spectral_r`) on a **single fixed scale matched
to the data** (the global min/max across all frames). That fixed, data-matched scale is what keeps the colours
comparable frame-to-frame *and* well-exposed — auto-ranging each frame (or using the full −40…40 climatology
scale) washes a summer field into a flat orange-red.

In [ ]:
%matplotlib inline
import os, sys
_wt = r"C:/python-environments/worktrees/cleopatra/perceptual-palettes/src"
if os.path.isdir(_wt) and _wt not in sys.path:
    sys.path.insert(0, _wt)
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import cleopatra
from cleopatra.array_glyph import ArrayGlyph, FrameLabel
from cleopatra.animation import embed_gif
print("cleopatra", cleopatra.__version__)

z = np.load(Path("../data/europe_t2m.npz"), allow_pickle=True)
celsius = z["celsius"].astype(float)          # (45, rows, cols) daily 2 m temperature, deg C
labels = list(z["labels"])
ext = [float(v) for v in z["extent"]]

STYLE = "temperature_2m"
# Widened past the data range so Spectral_r's pale midpoint sits at the cold edge, not on the summer data.
VMIN, VMAX = -15.0, 42.0
print("stack:", celsius.shape, "|", labels[0], "->", labels[-1],
      "| fixed scale", (round(VMIN, 1), round(VMAX, 1)), "degC")

## A single day — the warmest in the series

In [ ]:
day = int(np.argmax([np.nanmean(f) for f in celsius]))
glyph = ArrayGlyph(celsius[day], extent=[ext[0], ext[2], ext[1], ext[3]])
glyph.plot(style=STYLE, vmin=VMIN, vmax=VMAX, full_bleed=True)   # fills the frame, no white margin
plt.show()

## Animated over the full 45-day period

`full_bleed=True` fills the frame edge-to-edge — no chrome, and the masked seas read as a dark canvas rather
than white margin. Every frame shares the fixed data-matched scale, so the warming and cooling read directly
and nothing blows out.

In [ ]:
glyph = ArrayGlyph(celsius, extent=[ext[0], ext[2], ext[1], ext[3]])
anim = glyph.animate(labels, style=STYLE, vmin=VMIN, vmax=VMAX,
                     frame_label=FrameLabel(location=[ext[0] + 0.03 * (ext[1] - ext[0]),
                                                      ext[2] + 0.05 * (ext[3] - ext[2])],
                                            color="white"),
                     interval=140, full_bleed=True)
plt.close(glyph.fig)
embed_gif(anim, fps=7)

## The same series as a glowing plume over a **real basemap** — `temperature_flame`

A **single `ArrayGlyph.animate` call** does all of it now: the 45-frame loop, the `temperature_flame` preset on
the fixed `2…30 °C` scale, the date label, the **full-bleed** layout (`full_bleed=True` — fills the frame, no
chrome, aspect-matched so there's no distortion) and the **basemap** (`basemap=True` — a relief backdrop plus
Natural-Earth coastlines/borders, composed by `zorder`). The flame preset's value-linked opacity lets the cool
areas reveal the terrain while the hot plume glows on top (the ECMWF/CAMS look).

In [ ]:
FL_VMIN, FL_VMAX = 2.0, 30.0
west, east, south, north = ext
glyph = ArrayGlyph(celsius, extent=[west, south, east, north])
anim2 = glyph.animate(labels, style="temperature_flame", vmin=FL_VMIN, vmax=FL_VMAX,
                      add_colorbar=False, frame_label=FrameLabel(color="white"),
                      title="", interval=140, full_bleed=True, basemap=True)
plt.close(glyph.fig)
embed_gif(anim2, fps=7)